# A synthetic research lifecycle

This tutorial uses researchgraph 0.0.1, the v0.1 specification milestone, and the bundled `research` profile version 1.0. The specifications remain normative; this notebook describes the implemented API. See the [developer manual](developer-manual.md) for isolated installation and the local headless execution recipe, and the [user manual](user-manual.md) for concepts and commands. No notebook frontend is required.

We ask whether an explicitly constructed treatment adds two units to paired measurements. Standard-library synthetic data, a fixed seed, and explicit advancing whole-second UTC clocks make the exercise repeatable; generated graph identifiers are deliberately not fixed. There is no network, external dataset, scheduler, or reference runner. The quickstart's ungated `tasks` profile is a different example.

The outcome is a Question, Hypothesis, Prediction, agent-completed Task, saved Result, and human-approved Claim. **The positive shift is constructed, not scientific validation or evidence about a real intervention.** Three named review points use the same declared human actor. These are **simulated approvals** in an automated run, not authenticated identity or proof that a person reviewed the run.

In [ ]:
import hashlib
import json
import math
import os
import random
import statistics
import subprocess
import tempfile
from datetime import UTC, datetime, timedelta
from pathlib import Path

from researchgraph import Actor, Project, __version__
from researchgraph.profiles import Profile

# This is tutorial context, not a researchgraph core setting.
checkout_value = os.environ.get('RESEARCHGRAPH_CHECKOUT_ROOT')
if not checkout_value:
    raise RuntimeError('Set RESEARCHGRAPH_CHECKOUT_ROOT to the source checkout')
checkout = Path(checkout_value).expanduser().resolve(strict=True)
if not (checkout / 'pyproject.toml').is_file() or not (
    checkout / 'docs/research-tutorial.ipynb'
).is_file():
    raise RuntimeError('RESEARCHGRAPH_CHECKOUT_ROOT must identify this checkout')

parent = Path(tempfile.mkdtemp(prefix='researchgraph-tutorial-')).resolve()
if parent.is_relative_to(checkout):
    raise RuntimeError('Use an external TMPDIR: the demo must not be in the checkout')
demo = parent / 'demo'
if demo.exists() or demo.is_symlink():
    raise FileExistsError(f'Refusing existing demo: {demo}')
demo.mkdir()

# Ignore inherited Git overrides; identity/configuration belongs only to demo.
git_environment = {
    key: value for key, value in os.environ.items()
    if not key.startswith('GIT_') and key != 'PYTHONPATH'
}
git_environment['GIT_CONFIG_GLOBAL'] = str(parent / 'absent-global-gitconfig')
git_environment['GIT_CONFIG_NOSYSTEM'] = '1'


def git(*arguments):
    return subprocess.run(
        ['git', '-C', str(demo), *arguments],
        env=git_environment, check=True, capture_output=True, text=True, timeout=30,
    ).stdout.strip()


git('init')
git('config', '--local', 'user.name', 'Tutorial Git Identity')
git('config', '--local', 'user.email', 'tutorial@example.test')
git('config', '--local', 'commit.gpgsign', 'false')
# Core Git calls inherit the same explicitly isolated Git settings.
for key in tuple(os.environ):
    if key.startswith('GIT_'):
        del os.environ[key]
os.environ['GIT_CONFIG_GLOBAL'] = git_environment['GIT_CONFIG_GLOBAL']
os.environ['GIT_CONFIG_NOSYSTEM'] = '1'

human = Actor('human', 'simulated-reviewer', 'reviewer@example.test')
agent = Actor('agent', 'paired-experiment')
seed = 20261020
clock = datetime(2026, 10, 20, tzinfo=UTC)


def tick():
    global clock
    clock += timedelta(seconds=1)
    return clock


assert __version__ == '0.0.1'
assert Profile.load('research').version == '1.0'
assert Profile.load('tasks').version == '1.0'
with Project.init(demo, profile='research', actor=human, now=tick()) as project:
    assert project.profile_name == 'research'
    assert project.profile_version == '1.0'
print(f'Researchgraph {__version__}; spec milestone v0.1; research profile 1.0')
print(f'Retained project: {demo}')

## Question and review point 1: Hypothesis

Before approving, inspect the question, the narrowly phrased hypothesis, and the outgoing `answers` direction. We hypothesize only that this constructed treatment increases these synthetic measurements. The bundled research profile gates Hypothesis, Prediction, and Claim activation; **any declared human** may approve. Its declared `owner` and `reviewer` roles do not make these reviewer-only gates. Agents cannot satisfy them. Here the simulated reviewer approves explicitly, without `input()` or widgets.

In [ ]:
with Project.open(demo) as project:
    question = project.propose(
        'Question', 'Does our constructed treatment increase paired measurements?',
        actor=human, now=tick(),
    )
    hypothesis = project.propose(
        'Hypothesis', 'The constructed treatment increases the synthetic measurements',
        edges=[{'type': 'answers', 'to': question['id']}],
        actor=human, now=tick(),
    )
    assert hypothesis['status'] == 'proposed'
    hypothesis = project.approve(
        hypothesis['id'], note='Simulated review: inspect scope and answers edge',
        actor=human, now=tick(),
    )
    assert hypothesis['status'] == 'active'

## Review point 2: Prediction before measurement

Inspect the criterion before calculating results: 16 paired baselines from `random.Random(20261020).uniform(0, 10)`, each treated value equal to its baseline plus 2.0, and a mean paired delta close to 2.0. `expected` belongs to the outgoing `tests` edge from Prediction to Hypothesis, not a core statistical validator. The simulated human approves that explicit, deliberately trivial criterion.

In [ ]:
expected = 'For 16 constructed pairs, mean(treated - baseline) is 2.0 within floating tolerance'
with Project.open(demo) as project:
    prediction = project.propose(
        'Prediction', 'The mean paired increase will be two units',
        edges=[{
            'type': 'tests', 'to': hypothesis['id'],
            'fields': {'expected': expected},
        }],
        actor=human, now=tick(),
    )
    assert prediction['status'] == 'proposed'
    prediction = project.approve(
        prediction['id'], note='Simulated review: inspect prespecified expected criterion',
        actor=human, now=tick(),
    )
    task = project.propose(
        'Task', 'Generate and summarize the fixed-seed paired experiment',
        edges=[{'type': 'addresses', 'to': prediction['id']}],
        actor=agent, now=tick(),
    )
    running = project.claim(task['id'], lease_seconds=7200, actor=agent, now=tick())
    assert running['status'] == 'running'
    assert running['lease']['actor']['kind'] == 'agent'

## One explicit agent Task cycle

Task is the only claimable node type in this profile. Our distinct declared agent claims this one Task, computes the experiment, and completes its lease. This is not a scheduling loop. Generating measurements is ordinary consumer code, not field knowledge added to researchgraph. Completion records the summary and clears the lease; saving the Result artifact is a separate step.

In [ ]:
generator = random.Random(seed)
baselines = [generator.uniform(0, 10) for _ in range(16)]
pairs = [{'baseline': value, 'treated': value + 2.0} for value in baselines]
deltas = [pair['treated'] - pair['baseline'] for pair in pairs]
summary = {
    'seed': seed,
    'pairs': pairs,
    'sample_size': len(pairs),
    'expected_delta': 2.0,
    'mean_delta': statistics.mean(deltas),
}
assert summary['sample_size'] == 16
assert all(math.isfinite(value) for value in deltas)
assert all(math.isclose(value, 2.0, rel_tol=1e-12, abs_tol=1e-12) for value in deltas)
assert math.isclose(summary['mean_delta'], 2.0, rel_tol=1e-12, abs_tol=1e-12)
with Project.open(demo) as project:
    completed = project.complete(
        task['id'], 'Generated 16 constructed pairs; mean increase is two units',
        actor=agent, now=tick(),
    )
    assert completed['status'] == 'done'
    assert completed['lease'] is None

## Save real bytes on a Result

The outgoing `derived-from` edge records Result → completed Task. We first write nonempty JSON under `artifacts/<returned Result id>/experiment.json`, then call `attach` with that existing relative file path. **Attach records the file; it does not create or move it.** The Result, not merely the Task, owns the artifact record. Registration records its byte count, SHA-256, actor and time, and the normal committed write stages the named bytes.

In [ ]:
with Project.open(demo) as project:
    result = project.propose(
        'Result', 'Fixed-seed constructed paired measurements and summary',
        fields={'runs': 1, 'held_out': False},
        edges=[{'type': 'derived-from', 'to': task['id']}],
        actor=agent, now=tick(),
    )
    artifact_relative = Path('artifacts') / result['id'] / 'experiment.json'
    artifact_path = demo / artifact_relative
    artifact_path.parent.mkdir(parents=True, exist_ok=False)
    artifact_bytes = (json.dumps(summary, indent=2, sort_keys=True) + '\n').encode('utf-8')
    artifact_path.write_bytes(artifact_bytes)
    assert artifact_path.read_bytes() == artifact_bytes
    result = project.attach(
        result['id'], [artifact_relative.as_posix()], actor=agent, now=tick(),
    )
    assert artifact_path.read_bytes() == artifact_bytes
    record = result['artifacts'][0]
    assert record['bytes'] == len(artifact_bytes) > 0
    assert record['sha256'] == hashlib.sha256(artifact_bytes).hexdigest()

## Review point 3: evidence-backed Claim

Inspect the saved JSON and hashes, the prespecified criterion, the measured delta and the deliberately limited conclusion. Link Claim → Result with `cites` **before approval**, and Claim → Hypothesis with `supports`. The profile's `claim_needs_evidence` validator checks that a live outgoing cites relation exists; it does not judge scientific validity, artifact interpretation, or reviewer identity. Our simulated reviewer approves only the constructed demonstration. An approved Claim is **active**, not done.

In [ ]:
with Project.open(demo) as project:
    claim = project.propose(
        'Claim', 'The constructed treatment increased these synthetic pairs by two units',
        body='Illustrative construction only; no inference about a real intervention.',
        actor=agent, now=tick(),
    )
    cites = project.link(
        'cites', claim['id'], result['id'], actor=agent, now=tick(),
    )
    supports = project.link(
        'supports', claim['id'], hypothesis['id'], actor=agent, now=tick(),
    )
    assert claim['status'] == 'proposed'
    claim = project.approve(
        claim['id'], note='Simulated review: inspect Result bytes, criterion and limited conclusion',
        actor=human, now=tick(),
    )
    assert claim['status'] == 'active'

## Inspect graph, provenance, saved bytes and evidence

Reopen the retained project through the public API, using returned identifiers rather than golden IDs or commit hashes. These cells check the outcome; automated acceptance also reopens the project outside the notebook as an independent oracle. Low-level record reads include retired records; here we deliberately create exactly six nodes and six live edges. Evidence follows cites/derived-from to the Result and its Task and excludes the Claim itself. Every Project owner is closed by its context manager.

In [ ]:
with Project.open(demo) as project:
    assert project.profile_name == 'research' and project.profile_version == '1.0'
    nodes = project.get_nodes()
    assert len(nodes) == 6
    assert {node.type for node in nodes} == {
        'Question', 'Hypothesis', 'Prediction', 'Task', 'Result', 'Claim',
    }
    for identifier in (question['id'], hypothesis['id'], prediction['id'], result['id'], claim['id']):
        assert project.get_node(identifier).status.value == 'active'
    task_node = project.get_node(task['id'])
    assert task_node.status.value == 'done' and task_node.lease is None
    task_events = [event for event in task_node.approvals if event.what in {'claim', 'complete'}]
    assert [event.what for event in task_events] == ['claim', 'complete']
    assert all(event.actor == agent and event.actor.kind == 'agent' for event in task_events)
    assert agent.name != human.name and agent != human
    assert task_events[0].at < task_events[1].at

    approval_events = []
    for identifier in (hypothesis['id'], prediction['id'], claim['id']):
        approvals = [event for event in project.get_node(identifier).approvals if event.what == 'approve']
        assert len(approvals) == 1
        assert approvals[0].actor == human and approvals[0].actor.kind == 'human'
        approval_events.append(approvals[0])

    edges = project.get_edges()
    assert len(edges) == 6 and all(not edge.retired for edge in edges)
    assert {(edge.type, edge.from_, edge.to) for edge in edges} == {
        ('answers', hypothesis['id'], question['id']),
        ('tests', prediction['id'], hypothesis['id']),
        ('addresses', task['id'], prediction['id']),
        ('derived-from', result['id'], task['id']),
        ('cites', claim['id'], result['id']),
        ('supports', claim['id'], hypothesis['id']),
    }
    tests_relation = next(edge for edge in edges if edge.type == 'tests')
    assert tests_relation.fields['expected'] == expected
    cites_relation = next(edge for edge in edges if edge.type == 'cites')
    assert cites_relation.created < approval_events[-1].at

    saved_result = project.get_node(result['id'])
    assert saved_result.artifacts
    for artifact in saved_result.artifacts:
        relative = Path(artifact.path)
        assert relative.parts[:2] == ('artifacts', result['id'])
        path = (demo / relative).resolve()
        assert path.is_relative_to((demo / 'artifacts' / result['id']).resolve())
        data = path.read_bytes()
        assert len(data) == artifact.bytes > 0
        assert hashlib.sha256(data).hexdigest() == artifact.sha256
        assert saved_result.created < artifact.added_at < approval_events[-1].at
    restored = json.loads(artifact_path.read_bytes())
    assert restored['seed'] == seed and restored['sample_size'] == len(restored['pairs']) == 16
    restored_deltas = [pair['treated'] - pair['baseline'] for pair in restored['pairs']]
    assert math.isclose(restored['mean_delta'], statistics.mean(restored_deltas), abs_tol=1e-12)
    assert math.isclose(restored['mean_delta'], restored['expected_delta'], abs_tol=1e-12)

    evidence = project.evidence(claim['id'], actor=human, now=tick())
    evidence_by_id = {item['id']: item for item in evidence['items']}
    assert result['id'] in evidence_by_id and task['id'] in evidence_by_id
    assert claim['id'] not in evidence_by_id
    assert evidence_by_id[result['id']]['artifacts'] == [
        artifact.to_dict() for artifact in saved_result.artifacts
    ]
    timestamps = [node.created for node in nodes] + [edge.created for edge in edges]
    timestamps += [event.at for node in nodes for event in node.approvals]
    assert all(value.utcoffset() == timedelta(0) and value.microsecond == 0 for value in timestamps)
print('Verified simulated approvals, agent completion, Result bytes and Claim evidence.')
print(f'Retained project: {demo}')

## Retained inspection project

The printed absolute `<temporary parent>/demo` remains on disk for inspection; it is not part of the checkout. Re-execution creates a fresh parent rather than reusing prepared graph state. Remove that specific parent manually when finished inspecting it. No user/global kernelspec or Git identity was installed.

The notebook is stored without outputs, execution counts or runtime metadata. The documented headless recipe executes it in memory and does not overwrite this file. A successful run demonstrates declared identities, activation gates, task leases, artifact registration and evidence traversal—not authentication, genuine human review, or scientific validation. Files remain authoritative and the SQLite index is disposable.